# Broad Cell-Type Annotation by Label Transfer

Annotates the scVI-integrated snRNA-seq object (S4, all samples incl. liver biopsies)
into broad compartments. Logistic-regression predictions from the LuCA reference are
added and cross-tabulated against scVI Leiden clusters; clusters are labelled
manually from the predictions and TF-IDF markers, and ambiguous compartments are
sub-clustered and re-labelled. Outputs the annotated object used to split compartments.

In [ ]:
# !pip install --quiet scanpy leidenalg igraph scikit-learn
# %cd <project>/notebooks

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.feature_extraction.text import TfidfTransformer

## Config

In [ ]:
input_path = "./../snRNAseq/output/cellbender/S4_scVI_integrated_adata_cluster_latent10.h5ad"
prediction_path = "./../results/log_regression-cellbender/logreg_LUCAref_train_HVGs_20250605.csv"

output_path = "./../snRNAseq/output/cellbender/S5b_scVI_integrated_adata_cluster_latent10_logreg_major_cell_type.h5ad"
figures_output_dir = "./../figures-cellbender/integration/3_annotation/1_broad"
tables_output_dir = "./../results/annotation/1_broad"
seed = 0

cluster_key = "leiden_scvi"             # scVI Leiden clusters computed in S4 (resolution 0.6)
label_col = "cell_type"
sample_col = "sample"
use_rep = "X_scVI_n_latent_10"
prediction_cols = ["celltype_predictions", "probabilities"]

# TF-IDF marker ranking on binarised counts. False reproduces the original
# (TfidfTransformer fitted on genes x cells); True = standard cells-as-documents TF-IDF.
tfidf_cells_as_documents = False
tfidf_on_clusters = True                # ~15 min on 213k cells
tfidf_on_final_labels = True            # ~25 min on 213k cells
n_marker_genes = 10

# --- step 1: broad label per S4 Leiden cluster ---
broad_map = {
    "0": "Epithelial+Tumor cells", "1": "Epithelial+Tumor cells", "2": "Epithelial+Tumor cells",
    "3": "Alveolar cell type II", "4": "Myeloid", "5": "Alveolar cell type I",
    "6": "Epithelial+Tumor cells", "7": "Fibroblast", "8": "Lymphoid", "9": "Endothelial",
    "10": "Epithelial+Tumor cells+Doublets", "11": "Hepato", "12": "Club_Goblet cells",
    "13": "Ciliated epithelial", "14": "Epithelial+Tumor cells", "15": "Epithelial+Tumor cells",
    "16": "Epithelial+Tumor cells", "17": "Epithelial+Tumor cells", "18": "Myeloid",
}

# --- step 2: sub-cluster refinements, applied in order ---
# mapping keys are sub-cluster Leiden IDs; set mapping=None to explore a new parent first.
# Leiden IDs are only stable for the same data, scanpy/leidenalg versions and seed;
# the crosstab against predictions (section 4) is the check that IDs still match.
epi_markers = ["EPCAM", "CD14", "MMP1", "NAPSA", "SFTA2", "NKX2-1", "CEACAM6", "KRT5", "KRT14",
               "SCGB3A1", "SCGB3A2", "MUC5B", "MUC2", "MUC5AC", "BPIFB1", "FOXJ1"]
refinements = [
    dict(parent="Club_Goblet cells", resolution=0.6, markers=epi_markers,
         mapping={"0": "Club_Goblet cells", "1": "Epithelial+Tumor cells", "2": "Epithelial+Tumor cells",
                  "3": "Epithelial+Tumor cells", "4": "Club_Goblet cells", "5": "Epithelial+Tumor cells",
                  "6": "Club_Goblet cells", "7": "Club_Goblet cells", "8": "Epithelial+Tumor cells"}),
    dict(parent="Alveolar cell type II", resolution=0.6, markers=epi_markers + ["SFTPC", "SFTPB"],
         mapping={"0": "Alveolar cell type II", "1": "Epithelial+Tumor cells", "2": "Alveolar cell type II",
                  "3": "Alveolar cell type II", "4": "Alveolar cell type II", "5": "Alveolar cell type II",
                  "6": "Alveolar cell type II", "7": "Alveolar cell type II", "8": "Alveolar cell type II",
                  "9": "Alveolar cell type II"}),
    dict(parent="Lymphoid", resolution=0.6,
         markers=["CD4", "CD8A", "ENTPD1",
                  "SDC1", "MZB1", "CD38",                     # plasma
                  "CD19", "CD79A", "PAX5", "MS4A1",           # B cell (MS4A1 = CD20)
                  "PDGFRA", "FAP", "COL1A1", "KIT", "CPA3",   # fibroblast / mast
                  "CD68", "FCGR3A", "CD14"],                  # myeloid
         mapping={"0": "Lymphoid", "1": "Lymphoid", "2": "Lymphoid", "3": "B cell", "4": "Lymphoid",
                  "5": "Lymphoid", "6": "Plasma cell", "7": "Lymphoid", "8": "Lymphoid"}),
    dict(parent="Hepato", resolution=0.8,
         markers=["EPCAM", "NKX2-1", "NAPSA", "ALB", "APOA1", "TTR", "PTPRC", "n_genes", "percent_mito"],
         mapping={"0": "Epithelial+Tumor cells", "1": "Epithelial+Tumor cells", "2": "Epithelial+Tumor cells",
                  "3": "Epithelial+Tumor cells", "4": "Epithelial+Tumor cells", "5": "lowQC",
                  "6": "Epithelial+Tumor cells", "7": "Epithelial+Tumor cells", "8": "Epithelial+Tumor cells",
                  "9": "Doublets", "10": "Epithelial+Tumor cells", "11": "Doublets",
                  "12": "Epithelial+Tumor cells"}),
]
subcluster_n_neighbors = 30
subcluster_qc_cols = ["phase", "scrublet_doublet", "n_genes", "percent_mito"]

# --- final labels ---
drop_labels = []                        # e.g. ["lowQC", "Doublets", "Epithelial+Tumor cells+Doublets"]; [] reproduces the original
cell_type_colors = {
    "Club_Goblet cells": "#df3a82",
    "Alveolar cell type I": "#9e2f44",
    "Alveolar cell type II": "#d6bcc0",
    "Epithelial+Tumor cells": "#e996c3",
    "Ciliated epithelial": "#93019f",
    "Myeloid": "#7d87b9",
    "Lymphoid": "#a6a74e",
    "B cell": "#76b7b2",
    "Plasma cell": "#bab0ac",
    "Fibroblast": "#59a14f",
    "Endothelial": "#f28e2b",
    "Doublets": "grey",
    "lowQC": "grey",
    "Epithelial+Tumor cells+Doublets": "grey",
}
# canonical markers for the final sanity dotplot (genes absent from var are reported and skipped)
broad_markers = {
    "Epithelial": ["EPCAM", "KRT8", "NKX2-1", "NAPSA"],
    "AT1": ["AGER", "RTKN2"],
    "AT2": ["SFTPC", "SFTPB", "LAMP3"],
    "Club/Goblet": ["SCGB3A1", "SCGB3A2", "MUC5B", "BPIFB1"],
    "Ciliated": ["FOXJ1", "CAPS"],
    "Immune": ["PTPRC"],
    "Myeloid": ["CD68", "CD14", "FCGR3A"],
    "T/NK": ["CD3E", "CD4", "CD8A", "NKG7"],
    "B": ["MS4A1", "CD79A"],
    "Plasma": ["MZB1", "SDC1"],
    "Fibroblast": ["COL1A1", "PDGFRA", "FAP"],
    "Endothelial": ["PECAM1", "VWF"],
    "Hepatocyte": ["ALB", "APOA1"],
}

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 2
sc.set_figure_params(dpi=80, figsize=(8, 8))

## 1. Load data and add logistic-regression predictions

In [ ]:
adata = sc.read(input_path)
adata.uns.pop("log1p", None)
assert np.allclose(adata.X[:1000].data, np.round(adata.X[:1000].data)), "X is expected to hold raw counts"

prediction = pd.read_csv(prediction_path, index_col=0)
missing = adata.obs_names.difference(prediction.index)
print(f"predictions for {adata.n_obs - len(missing)}/{adata.n_obs} cells; missing: {len(missing)}")
for col in prediction_cols:
    adata.obs[col] = prediction[col].reindex(adata.obs_names)
adata.obs["celltype_predictions"] = adata.obs["celltype_predictions"].astype("category")
adata.obs["scrublet_doublet"] = adata.obs["is_doublet"].astype(str).astype("category")

adata

In [ ]:
sc.pl.umap(adata, color=["probabilities", "celltype_predictions"], size=15,
           save="_scVI_latent10_labTransfering_all.pdf")
sc.pl.umap(adata, color="celltype_predictions", legend_loc="on data", legend_fontsize="xx-small",
           size=15, save="_scVI_latent10_labTransferOndata.pdf")
sc.pl.umap(adata, color=[cluster_key, "scrublet_doublet", "scrublet_score", "n_genes"], ncols=4, size=10,
           legend_loc="on data", color_map="OrRd", save="_leiden_scvi_ondata_all.pdf")

## 2. Marker and prediction helpers

`tfidf_markers` binarises counts, applies TF-IDF and ranks genes (Wilcoxon).
`prediction_crosstab` gives, per cluster, the share of each predicted label; this is the
label-transfer evidence behind each manual assignment.

In [ ]:
def tfidf_markers(adata, groupby, method="wilcoxon", cells_as_documents=tfidf_cells_as_documents):
    """Rank genes per group on TF-IDF-transformed binarised counts (X must hold counts)."""
    X = sparse.csr_matrix(adata.X)
    X_bin = (X > 0).astype(np.float32)
    if cells_as_documents:
        tfidf = TfidfTransformer().fit_transform(X_bin)
    else:
        tfidf = TfidfTransformer().fit_transform(X_bin.T).T
    bdata = sc.AnnData(X=sparse.csr_matrix(tfidf), obs=adata.obs[[groupby]].copy(), var=adata.var[[]].copy())
    bdata.obs[groupby] = bdata.obs[groupby].astype("category").cat.remove_unused_categories()
    sc.tl.rank_genes_groups(bdata, groupby=groupby, use_raw=False, method=method)
    return bdata


def prediction_crosstab(obs, groupby, top=3):
    """Top predicted labels (fraction of cells) per group."""
    ct = pd.crosstab(obs[groupby], obs["celltype_predictions"], normalize="index")
    out = {}
    for group, row in ct.iterrows():
        row = row.sort_values(ascending=False).head(top)
        out[group] = "; ".join(f"{k} ({v:.0%})" for k, v in row.items() if v > 0)
    summary = pd.DataFrame({"n_cells": obs[groupby].value_counts(), "top_predictions": pd.Series(out)})
    summary["median_probability"] = obs.groupby(groupby, observed=True)["probabilities"].median()
    return summary


def check_genes(genes, var_names, label=""):
    """Split a panel into genes/obs columns present and missing."""
    present = [g for g in genes if g in var_names]
    missing = [g for g in genes if g not in var_names]
    if missing:
        print(f"{label}: not found, skipped -> {missing}")
    return present

## 3. Label transfer at S4 cluster level

In [ ]:
cluster_summary = prediction_crosstab(adata.obs, cluster_key)
cluster_summary["broad_label"] = cluster_summary.index.map(broad_map)
cluster_summary.to_csv(f"{tables_output_dir}/leiden_scvi_prediction_summary.csv")
cluster_summary

In [ ]:
if tfidf_on_clusters:
    tfidf_clusters = tfidf_markers(adata, cluster_key)
    sc.pl.rank_genes_groups_dotplot(tfidf_clusters, n_genes=n_marker_genes, dendrogram=False,
                                    standard_scale="var", save="tfidf_leiden_scvi.pdf")

In [ ]:
unmapped = set(adata.obs[cluster_key].astype(str)) - set(broad_map)
assert not unmapped, f"clusters without a broad label: {sorted(unmapped)}"
adata.obs[label_col] = adata.obs[cluster_key].astype(str).map(broad_map).astype("category")

sc.pl.umap(adata, color=["scrublet_doublet", label_col], size=15, save="_cell_type_broad_step1.pdf")
adata.obs[label_col].value_counts()

## 4. Sub-cluster refinement

Each parent label is re-clustered on the scVI latent space and inspected (QC, logreg and
TF-IDF markers, marker UMAPs, prediction crosstab). The loop restarts from `broad_map`,
so re-running the cell does not compound earlier refinements.

In [ ]:
def refine(adata, parent, resolution, markers, mapping=None, label_col=label_col):
    """Sub-cluster cells labelled `parent`, plot evidence, and relabel with `mapping`."""
    tag = parent.replace(" ", "_").replace("/", "_")
    sub = adata[adata.obs[label_col] == parent].copy()
    if sub.n_obs <= subcluster_n_neighbors:
        raise ValueError(f"{parent}: {sub.n_obs} cells, too few to sub-cluster")
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)
    sc.pp.neighbors(sub, n_neighbors=subcluster_n_neighbors, use_rep=use_rep, random_state=seed)
    sc.tl.umap(sub, random_state=seed)
    sc.tl.leiden(sub, key_added="leiden_sub", resolution=resolution, flavor="leidenalg", random_state=seed)

    sc.pl.umap(sub, color=["leiden_sub"] + subcluster_qc_cols, ncols=3, wspace=0.3, size=15,
               color_map="OrRd", legend_loc="on data", save=f"_{tag}_subclusters_qc.pdf")
    sc.tl.rank_genes_groups(sub, "leiden_sub", method="logreg", key_added="logreg", max_iter=1000)
    sc.pl.rank_genes_groups_dotplot(sub, n_genes=n_marker_genes, key="logreg", groupby="leiden_sub",
                                    standard_scale="var", dendrogram=False, save=f"{tag}_logreg.pdf")
    counts = sc.AnnData(X=adata[sub.obs_names].X, obs=sub.obs[["leiden_sub"]], var=adata.var[[]])
    sc.pl.rank_genes_groups_dotplot(tfidf_markers(counts, "leiden_sub"), n_genes=n_marker_genes,
                                    dendrogram=False, standard_scale="var", save=f"{tag}_tfidf.pdf")
    panel = check_genes(markers, set(sub.var_names) | set(sub.obs.columns), parent)
    sc.pl.umap(sub, color=["leiden_sub"] + panel + ["celltype_predictions"], use_raw=False,
               color_map="OrRd", size=55, legend_fontsize="small", save=f"_{tag}_markers.pdf")

    summary = prediction_crosstab(sub.obs, "leiden_sub")
    if mapping is not None:
        unmapped = set(sub.obs["leiden_sub"].astype(str)) - set(mapping)
        assert not unmapped, f"{parent}: sub-clusters without a label: {sorted(unmapped)}"
        new_labels = sub.obs["leiden_sub"].astype(str).map(mapping)
        summary["new_label"] = summary.index.map(mapping)
        labels = adata.obs[label_col].astype(str)
        labels.loc[new_labels.index] = new_labels
        adata.obs[label_col] = labels.astype("category")
    summary.to_csv(f"{tables_output_dir}/{tag}_subcluster_summary.csv")
    display(summary)
    return sub

In [ ]:
adata.obs[label_col] = adata.obs[cluster_key].astype(str).map(broad_map).astype("category")
subsets = {}
for r in refinements:
    print(f"=== {r['parent']} ===")
    subsets[r["parent"]] = refine(adata, **r)

sc.pl.umap(adata, color=["scrublet_doublet", label_col], size=15, save="_cell_type_broad_refined.pdf")
pd.crosstab(adata.obs[cluster_key], adata.obs[label_col])

## 5. Final labels, colours and sanity checks

In [ ]:
if drop_labels:
    adata = adata[~adata.obs[label_col].isin(drop_labels)].copy()
adata.obs[label_col] = adata.obs[label_col].cat.remove_unused_categories()

labels = list(adata.obs[label_col].cat.categories)
no_color = [l for l in labels if l not in cell_type_colors]
assert not no_color, f"labels without a colour: {no_color}"
adata.uns[f"{label_col}_colors"] = [cell_type_colors[l] for l in labels]

sc.pl.umap(adata, color=label_col, size=15, legend_loc="on data", legend_fontsize="x-small",
           legend_fontoutline=True, save="_cell_type_final.pdf")

final_summary = prediction_crosstab(adata.obs, label_col)
final_summary.to_csv(f"{tables_output_dir}/cell_type_prediction_summary.csv")
final_summary

In [ ]:
marker_panel = {k: check_genes(v, adata.var_names, k) for k, v in broad_markers.items()}
marker_panel = {k: v for k, v in marker_panel.items() if v}

tmp = adata.copy()
sc.pp.normalize_total(tmp, target_sum=1e4)
sc.pp.log1p(tmp)
sc.pl.dotplot(tmp, marker_panel, groupby=label_col, standard_scale="var", save="cell_type_broad_markers.pdf")
del tmp

if tfidf_on_final_labels:
    tfidf_final = tfidf_markers(adata, label_col)
    sc.pl.rank_genes_groups_dotplot(tfidf_final, n_genes=n_marker_genes, dendrogram=False,
                                    standard_scale="var", save="tfidf_cell_type.pdf")

In [ ]:
def proportion_barplot(adata, label_col, group_col, colors, fname):
    """Stacked horizontal bars: % of each label per group."""
    props = pd.crosstab(adata.obs[group_col], adata.obs[label_col], normalize="index") * 100
    props = props.iloc[::-1]
    ax = props.plot.barh(stacked=True, edgecolor="none", figsize=(4, 0.25 * len(props) + 1),
                         color=[colors[c] for c in props.columns], grid=False, width=0.85)
    ax.set_xlim(0, 100)
    ax.set_xlabel(f"{label_col} (%)")
    ax.set_ylabel(group_col)
    ax.legend(bbox_to_anchor=(1, 1), frameon=False, fontsize=8)
    plt.savefig(f"{figures_output_dir}/{fname}", bbox_inches="tight", dpi=300)
    plt.show()
    return props


props = proportion_barplot(adata, label_col, sample_col, cell_type_colors,
                           f"barplot_{sample_col}_proportions_{label_col}.pdf")
props.round(2).to_csv(f"{tables_output_dir}/{label_col}_proportions_per_{sample_col}.csv")

## 6. Save (X = raw counts)

In [ ]:
adata.write(output_path)
print(f"total cells: {adata.n_obs}")
adata.obs[label_col].value_counts()